# CI/CD 第3周：Docker & 部署 — 从镜像构建到 K8s 部署

> **学习目标**：在 CI/CD 中构建 Docker 镜像、推送到 Registry、部署到 Kubernetes

---

## 把 Docker 和 CI/CD 结合起来

前两周的 CI/CD 只做了"代码检查 + 测试"——这是 CI 的部分。

真正的**持续交付（CD）** 需要把代码变成可运行的**产物**，然后部署到环境中。

Docker 镜像就是最理想的交付产物：
- **不可变**：构建一次，到处运行
- **自包含**：包含应用 + 环境，无依赖缺失
- **版本化**：通过 tag 管理版本
- **标准化**：OCI 标准，任何平台都支持

```
代码提交 → 自动测试 → 构建镜像 → 推送 Registry → 部署到 K8s
                                   ↑
                              这周的重点
```

---

## Day 15：在 Actions 中构建 Docker 镜像

### 需要的 Action

| Action | 用途 |
|--------|------|
| `docker/login-action@v3` | 登录到容器 Registry（GHCR / Docker Hub） |
| `docker/build-push-action@v6` | 构建并推送镜像 |
| `docker/metadata-action@v5` | 生成镜像 tag 和 label |

### 基本构建 Workflow

```yaml
name: Build Docker Image
on: push

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: Build Docker image
        uses: docker/build-push-action@v6
        with:
          context: .           # Dockerfile 所在目录
          push: false          # 仅构建，不推送
          tags: my-app:latest
          load: true           # 将镜像加载到本地 Docker 守护进程
```

**注意**：`push: false` + `load: true` 构建后镜像会在 Runner 本地可用，但不会推送到远程 Registry。

### 验证构建结果

构建完成后，可以在 Runner 上运行 `docker images` 查看，或者用 `docker run` 测试。

In [ ]:
# 模拟 Docker 构建流程

import time

print("=" * 60)
print("🐳 Docker 构建流程模拟")
print("=" * 60)

steps = [
    ("actions/checkout@v4", "拉取代码到 Runner"),
    ("docker/setup-buildx-action@v3", "设置 Buildx（支持多平台构建）"),
    ("检查 Dockerfile", "确认 Dockerfile 存在"),
    ("docker build -t my-app", "构建镜像"),
]

print("\n📋 Workflow: Build Docker Image (ONLY BUILD, NO PUSH)")
print("-" * 40)

for step_name, desc in steps:
    print(f"\n  ▶ Step: {step_name}")
    print(f"     {desc}")
    time.sleep(0.3)
    print("     ✅ 完成")

print("\n" + "-" * 40)
print("✅ 镜像构建成功！")
print("   镜像名: my-app:latest")
print("   大小: ~180MB")
print("   可执行: docker run --rm my-app:latest")

print("\n⚠️ 注意：push: false 时镜像只存在于 Runner 本地")
print("   如果要在其他机器使用，需要 push: true")

### docker/build-push-action 关键参数

| 参数 | 说明 | 示例 |
|------|------|------|
| `context` | 构建上下文目录 | `.` |
| `dockerfile` | Dockerfile 路径（默认 context/Dockerfile） | `./Dockerfile.prod` |
| `push` | 是否推送 | `true` / `false` |
| `load` | 是否加载到本地 dockerd | `true` / `false` |
| `tags` | 镜像 tag 列表 | `app:v1,app:latest` |
| `build-args` | 构建参数 | `VERSION=1.0` |
| `cache-from` | 缓存来源 | `type=gha` / `type=registry,ref=...` |
| `cache-to` | 缓存目标 | 同上 |
| `platforms` | 目标平台 | `linux/amd64,linux/arm64` |

### 练习

写一个 Workflow：checkout → build Docker 镜像（只构建，不推送），确认构建成功。

---

## Day 16：推送到 Registry

### 选择 Registry

| Registry | 地址 | 说明 |
|----------|------|------|
| GitHub Container Registry (GHCR) | `ghcr.io` | 与 GitHub 集成最好，用 GITHUB_TOKEN |
| Docker Hub | `docker.io` | 最流行，但需要 Docker ID |
| 阿里云 ACR | `registry.cn-hangzhou.aliyuncs.com` | 国内速度快 |

### 推送到 GHCR

```yaml
- name: Login to GHCR
  uses: docker/login-action@v3
  with:
    registry: ghcr.io
    username: \${{ github.actor }}
    password: \${{ secrets.GITHUB_TOKEN }}

- name: Build and push
  uses: docker/build-push-action@v6
  with:
    push: true
    tags: ghcr.io/\${{ github.repository }}:latest
```

**关键点**：
- `GITHUB_TOKEN` 是由 GitHub 自动生成的临时 Token，不需要手动配置
- 但是需要在仓库 Settings → Actions → General 中勾选 "Read and write permissions"
- 或者在 workflow 中显式设置 permissions：

```yaml
permissions:
  contents: read
  packages: write  # 允许推送包到 GHCR
```

In [ ]:
# 模拟 GHCR 推送流程

import time

print("=" * 60)
print("🐳 构建 + 推送镜像到 GHCR")
print("=" * 60)

print("\n1️⃣  登录到 GHCR")
print("   docker/login-action@v3")
print("   registry: ghcr.io")
print("   username: \${{ github.actor }}")
print("   password: \${{ secrets.GITHUB_TOKEN }}")
time.sleep(0.3)
print("   ✅ 登录成功")

print("\n2️⃣  Build and Push")
print("   docker/build-push-action@v6")
print("   context: .")
print("   push: true")
print("   tags: |")
print("     ghcr.io/my-org/my-app:latest")
time.sleep(0.5)
print("   ✅ 构建成功")
print("   ✅ 推送成功")

print("\n3️⃣  推送结果")
print("   📦 ghcr.io/my-org/my-app:latest")
print("   📦 Image digest: sha256:a1b2c3d4e5f6...")
print("   📦 Size: ~180MB")

print("\n📖 GHCR 包地址:")
print("   https://github.com/orgs/my-org/packages")

### 练习

在 Workflow 中构建镜像并推送到 GHCR。需要：
1. 确保仓库 Settings → Actions → General → Workflow permissions 设为 "Read and write permissions"
2. Workflow 中添加 permissions: `packages: write`
3. 使用 `docker/login-action@v3` 和 `docker/build-push-action@v6`

```yaml
name: Build and Push to GHCR
on: push
permissions:
  contents: read
  packages: write
jobs:
  push:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: \${{ github.actor }}
          password: \${{ secrets.GITHUB_TOKEN }}
      - uses: docker/build-push-action@v6
        with:
          push: true
          tags: ghcr.io/\${{ github.repository }}:latest
```

---

## Day 17：镜像 Tag 策略

### 为什么需要 Tag 策略？

`latest` 是不够的——你得能回溯到任意一个历史版本的镜像。

### 常见的 Tag 策略

```yaml
# 使用 docker/metadata-action 自动生成 tags
- name: Extract metadata
  id: meta
  uses: docker/metadata-action@v5
  with:
    images: ghcr.io/\${{ github.repository }}
    tags: |
      type=sha,format=short    # commit SHA 短格式
      type=ref,event=branch    # 分支名
      type=semver,pattern=v{{version}}  # 语义版本（tag push）
      type=raw,value=latest    # latest
```

**各策略说明**：

| Tag 策略 | 例子 | 使用场景 |
|----------|------|----------|
| `latest` | `latest` | 开发环境，永远最新 |
| commit SHA | `a1b2c3d` | 每个 commit 唯一，可回溯 |
| 分支名 | `main`, `dev` | 区分不同分支的构建 |
| 语义版本 | `v1.2.3` | 正式发布版本 |

**完整的 tag 示例**（一次构建产生多个 tag）：

```
ghcr.io/my-org/my-app:latest
ghcr.io/my-org/my-app:main
ghcr.io/my-org/my-app:a1b2c3d
ghcr.io/my-org/my-app:v1.2.3     # 仅 tag push 时
```

In [ ]:
# 模拟 Tag 策略

import time

print("=" * 60)
print("🏷️ 镜像 Tag 策略")
print("=" * 60)

# 模拟一次 main 分支 push
print("\n场景 1: main 分支 push")
print("  Commit: a1b2c3d "feat: add login"")
print("\n  生成 Tags:")
print("    - ghcr.io/my-org/my-app:latest")
print("    - ghcr.io/my-org/my-app:main")
print("    - ghcr.io/my-org/my-app:a1b2c3d")

# 模拟 tag push
print("\n场景 2: 发布 tag v2.1.0")
print("  Tag: v2.1.0")
print("  Commit: e5f6a7b "release: v2.1.0"")
print("\n  生成 Tags:")
print("    - ghcr.io/my-org/my-app:v2.1.0")
print("    - ghcr.io/my-org/my-app:latest")
print("    - ghcr.io/my-org/my-app:e5f6a7b")

# 模拟 dev 分支
print("\n场景 3: dev 分支开发")
print("  Branch: dev")
print("  Commit: c8d9e0f "WIP: working on feature"")
print("\n  生成 Tags:")
print("    - ghcr.io/my-org/my-app:dev")
print("    - ghcr.io/my-org/my-app:c8d9e0f")

print("\n" + "-" * 40)
print("✅ 好处:")
print("  - latest: 方便开发环境使用")
print("  - commit SHA: 精确定位每个版本")
print("  - semver: 正式发布的里程碑")

### 练习

给 Workflow 加上三类 tag：`latest`（main push）、`commit-<sha>`（每次 push）、`<tag>`（tag push 触发时）。

---

## Day 18：部署到 K8s（基础）

### 前置知识

Kubernetes（K8s）是目前最流行的容器编排平台。在 CI/CD 中部署到 K8s 的典型流程：

```
构建镜像 → 推送 Registry → 更新 Deployment 镜像 → kubectl apply
```

### 需要的准备

1. **K8s 集群**：可用 minikube / kind 本地搭建，或使用云服务（AKS / EKS / TKE）
2. **kubeconfig**：连接集群的配置文件，存入 Secrets
3. **K8s 资源 YAML**：Deployment + Service 的定义

### Workflow 中的部署步骤

```yaml
- name: Configure kubectl
  run: |
    mkdir -p \$HOME/.kube
    echo "\${{ secrets.KUBE_CONFIG }}" > \$HOME/.kube/config

- name: Deploy to K8s
  run: |
    kubectl set image deployment/my-app \
      my-app=ghcr.io/\${{ github.repository }}:\${{ github.sha }}
    kubectl rollout status deployment/my-app
```

In [ ]:
# 模拟 K8s 部署流程

import time

print("=" * 60)
print("☸️ K8s 部署流程模拟")
print("=" * 60)

steps = [
    ("拉取 kubeconfig", "从 Secrets 读取 KUBE_CONFIG"),
    ("配置 kubectl", "写入 ~/.kube/config"),
    ("更新镜像版本", "kubectl set image deployment/my-app my-app=ghcr.io/x/app:a1b2c3d"),
    ("触发滚动更新", "kubectl rollout restart deployment/my-app"),
    ("监控部署状态", "kubectl rollout status deployment/my-app"),
]

print("\n📋 Deployment Pipeline")
print("-" * 40)
for step_name, desc in steps:
    print(f"\n  ▶ {step_name}")
    print(f"     {desc}")
    time.sleep(0.3)
    print("     ✅ 完成")

print("\n" + "-" * 40)
print("✅ 部署成功!")
print("   新版本: a1b2c3d")
print("   副本数: 3/3")
print("   状态: Ready")

print("\n📊 rollout status:")
print("   Waiting for deployment "my-app" rollout to finish: 0 of 3 updated replicas are available...")
print("   Waiting for deployment "my-app" rollout to finish: 1 of 3 updated replicas are available...")
print("   Waiting for deployment "my-app" rollout to finish: 2 of 3 updated replicas are available...")
print("   ✅ deployment "my-app" successfully rolled out")

### 练习

1. 创建一个 K8s Deployment + Service YAML
2. 在 CI 中把新镜像版本写入 manifest
3. 执行 `kubectl apply` 更新集群

---

## Day 19：多环境部署（dev / staging / prod）

### Environment 的概念

GitHub Environments 让你为不同部署环境管理独立的配置和审批规则：

```
Settings → Environments → 创建 dev / staging / prod
```

每个 Environment 可以设置：
- **Secrets**：不同环境的密钥（如 dev 和 prod 用不同的 `KUBE_CONFIG`）
- **Required reviewers**：需要哪些人审批才能部署
- **Wait timer**：审批前等待多久
- **Branch restriction**：哪些分支可以部署到此环境

### Workflow 中的使用

```yaml
jobs:
  deploy-dev:
    environment: dev
    runs-on: ubuntu-latest
    steps:
      - name: Deploy to dev
        run: |
          kubectl set image deployment/my-app \
            my-app=ghcr.io/x/app:\${{ github.sha }}

  deploy-staging:
    needs: deploy-dev
    environment: staging  # 可能需要审批
    runs-on: ubuntu-latest
    steps:
      - run: ./deploy.sh staging
```

**环境映射约定**：

| 分支 | 环境 | 触发方式 |
|------|------|----------|
| `main` | dev | 自动部署 |
| `release/*` | staging | 自动部署 |
| `v*` tag | prod | 需要审批 + 手动触发 |

In [ ]:
# 模拟多环境部署

import time

print("=" * 60)
print("🏗️ 多环境部署流程")
print("=" * 60)

print("\n📋 部署策略:")
print("   main 分支 → dev (自动)")
print("   release/* 分支 → staging (自动)")
print("   v* tag → prod (需要审批)")

print("\n" + "-" * 40)
print("🔹 场景: 发布 v2.1.0")
print("-" * 40)

# Dev 部署
print("\n1️⃣  Dev 部署 (自动) [main branch]")
print("   环境: dev")
time.sleep(0.3)
print("   镜像: ghcr.io/my-org/my-app:a1b2c3d")
print("   状态: ✅ 部署成功")

# Staging 部署
print("\n2️⃣  Staging 部署 (自动) [release/v2.1.0 branch]")
print("   环境: staging")
time.sleep(0.3)
print("   镜像: ghcr.io/my-org/my-app:v2.1.0")
print("   状态: ✅ 部署成功")

# Prod 部署
print("\n3️⃣  Production 部署 (需要审批) [tag: v2.1.0]")
print("   环境: prod")
print("   审批人: @tech-lead @ops-manager")
print("   ⏳ 等待审批...")
time.sleep(0.5)
print("   ✅ @tech-lead 已批准")
print("   ✅ @ops-manager 已批准")
print("   镜像: ghcr.io/my-org/my-app:v2.1.0")
print("   策略: RollingUpdate (滚动更新，无停机)")
print("   状态: ✅ 部署成功")

print("\n" + "=" * 60)
print("🎉 v2.1.0 已部署到所有环境！")
print("=" * 60)

### 练习

创建 dev 和 staging 两个 Environment，配置各自的 `KUBE_CONFIG` Secret，Workflow 根据分支部署到不同环境：
- `main` → dev（自动）
- tag push（`v*`）→ staging（自动）

---

## Day 20：Docker Layer 缓存加速

### 为什么需要 Layer 缓存？

每次 CI 构建 Docker 镜像都是**从头构建**的——所有层都重新生成。对于依赖多的项目，这非常耗时。

**Docker Layer Cache** 可以复用上一次构建的层，只重新构建变更的部分。

### 两种缓存方式

#### 1. GitHub Actions Cache (GHA)

```yaml
- uses: docker/build-push-action@v6
  with:
    cache-from: type=gha
    cache-to: type=gha,mode=max
```

- **优点**：不需要额外的 Registry，用 GitHub 的缓存存储
- **缺点**：缓存只能在同一个仓库的 workflow 中复用
- **适用**：单仓库项目

#### 2. Registry Cache

```yaml
- uses: docker/build-push-action@v6
  with:
    cache-from: type=registry,ref=ghcr.io/\${{ github.repository }}:cache
    cache-to: type=registry,ref=ghcr.io/\${{ github.repository }}:cache,mode=max
```

- **优点**：缓存存在 Registry 中，全局可用
- **缺点**：需要 Registry 存储空间
- **适用**：多仓库引用同一缓存

### 性能对比

| 场景 | 首次构建 | 代码变更 | 仅依赖变更 |
|------|----------|----------|------------|
| 无缓存 | 180s | 180s | 180s |
| GHA cache | 180s | 15s | 120s |
| Registry cache | 180s | 15s | 90s |

In [ ]:
# 模拟 Layer 缓存效果对比

import time

print("=" * 60)
print("⚡ Docker Layer 缓存性能对比")
print("=" * 60)

print("\n📊 测试项目: 一个 Python Web 应用")
print("   - 基础镜像: python:3.12-slim (~150MB)")
print("   - 依赖: Flask, SQLAlchemy, Pydantic, etc.")
print("   - 代码: 500KB Python 源代码")
print("")

scenarios = [
    ("无缓存", 180, 180, 180, "❌"),
    ("GHA cache backend", 185, 15, 120, "✅"),
    ("Registry cache", 185, 15, 90, "✅"),
]

print(f"{'缓存方式':<20} {'首次构建':<12} {'改代码':<12} {'改依赖':<12}")
print("-" * 56)
for name, first, code, dep, icon in scenarios:
    print(f"{icon} {name:<18} {first:<12} {code:<12} {dep:<12}")

print("\n" + "-" * 40)
print("💡 关键观察:")
print("  - 首次构建: 有缓存的反而略慢（保存缓存也需要时间）")
print("  - 改代码: 缓存极大加速（只重建应用层）")
print("  - 改依赖: Registry cache 更快（layer 存储在 Registry，拉取快）")
print("  - GHA cache: 不用额外配置，适合大多数项目")

### 练习

对比三种场景的构建时间：无缓存、GHA cache backend、Registry cache。

---

## 🎯 第3周总结

### 核心技能回顾

| 技能 | 关键 Action / 命令 |
|------|-------------------|
| 构建镜像 | `docker/build-push-action@v6` |
| 登录 Registry | `docker/login-action@v3` |
| Tag 策略 | `docker/metadata-action@v5` |
| 层缓存 | `cache-from: type=gha` / `type=registry` |
| K8s 部署 | `kubectl set image` + `kubectl rollout status` |
| 多环境 | `environment:` Job 级别指定 |

### 关键配置模板

```yaml
permissions:
  contents: read
  packages: write

jobs:
  build-and-push:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: \${{ github.actor }}
          password: \${{ secrets.GITHUB_TOKEN }}
      - uses: docker/metadata-action@v5
        id: meta
        with:
          images: ghcr.io/\${{ github.repository }}
          tags: |
            type=sha,format=short
            type=ref,event=branch
            type=raw,value=latest
      - uses: docker/build-push-action@v6
        with:
          push: true
          tags: \${{ steps.meta.outputs.tags }}
          cache-from: type=gha
          cache-to: type=gha,mode=max
```

---

## 🧪 综合练习：完整的"代码提交 → 镜像 → 部署"流水线

构建一个完整的 CI/CD 流水线，将前两周的知识和大本周的 Docker + 部署结合起来：

```
push main / PR
  ├── Job: test (matrix: 3.10/3.11/3.12, pip cache)
  ├── Job: lint
  │
  └── Job: build-and-push (依赖 test + lint)
        ├── 构建 Docker 镜像（layer 缓存）
        ├── tag: latest / commit-sha / branch-name
        └── push to GHCR

tag push (v*)
  └── Job: deploy-staging (依赖 build-and-push, environment: staging)
        ├── 拉取 KUBE_CONFIG from secrets
        ├── kubectl set image → 更新 staging 集群
        └── kubectl rollout status → 等待部署完成
```

In [ ]:
# 模拟完整的 CI/CD 流水线

import time

print("=" * 70)
print("🚀 完整 CI/CD Pipeline: Code → Image → Deploy")
print("=" * 70)

pipeline = [
    # Phase 1: Test & Lint (并行)
    ("Phase 1", "test (3.10, ubuntu)", None),
    ("Phase 1", "test (3.11, ubuntu)", None),
    ("Phase 1", "test (3.12, ubuntu)", None),
    ("Phase 1", "lint (ruff + mypy)", None),

    # Phase 2: Build & Push (依赖 Phase 1)
    ("Phase 2", "build-and-push (Docker)", "Phase 1"),

    # Phase 3: Deploy (仅 tag push)
    ("Phase 3", "deploy-staging (K8s)", "Phase 2"),
]

print("\n📋 Pipeline 结构:")
print('''
    test ─┐
    test ─┤
    test ─┤               deploy-staging (仅 tag push, 需审批)
    lint ─┤→ build-and-push ─→ ┌─────────────────────┐
                               │ environment: staging │
                               │ kubectl apply ...    │
                               └─────────────────────┘
''')

completed_phases = set()

for phase, job, dependency in pipeline:
    if dependency and dependency not in completed_phases:
        print(f"\n⏳ '{job}' 等待 '{dependency}' 完成...")
        time.sleep(0.2)

    print(f"\n▶ [{phase}] {job}")
    time.sleep(0.4)
    print(f"  ✅ 完成")

    if "test" in job:
        print(f"     pytest: 42 passed, coverage: 87%")
    elif "lint" in job:
        print(f"     ruff: 0 errors, mypy: 0 errors")
    elif "build" in job:
        print(f"     Tags: latest, a1b2c3d, main")
        print(f"     Pushed to: ghcr.io/my-org/my-app")
    elif "deploy" in job:
        print(f"     namespace: staging")
        print(f"     rollout: 3/3 replicas updated")
        print(f"     🎉 部署完成!")

print("\n" + "=" * 70)
print("🎉 完整流水线执行成功！")
print("=" * 70)